In [6]:
import sys
import os

sys.path.append(os.path.abspath(".."))
import pandas as pd
from config import PATH_PROCESSED, PATH_SPLITS, PATH_PREDICTIONS, PATH_CALIBRATED, COHORTS, ALL_MODELS, ALL_METHODS, N_FOLDS
import pickle
import os
os.chdir("..")
import numpy as np

dfs = {c: pd.read_csv(f"{PATH_PROCESSED}{c}_clean.csv") for c in COHORTS}
col_sets = {c: set(df.columns) for c, df in dfs.items()}

reference = col_sets[COHORTS[0]]
all_match = True
for c in COHORTS[1:]:
    diff = col_sets[c].symmetric_difference(reference)
    if diff:
        print(f"MISMATCH in {c}: {diff}")
        all_match = False

print("Schema check:", "PASSED - all 4 cohorts have matching columns" if all_match else "FAILED - see mismatches above")

MISMATCH in child: {'ethnicity_middle eastern', "country_of_res_'United Arab Emirates'", "ethnicity_'South Asian'", "country_of_res_'New Zealand'", 'ethnicity_White European', 'ethnicity_south asian', 'ethnicity_black', 'country_of_res_India', 'ethnicity_White-European', 'country_of_res_Australia', 'country_of_res_Unknown', "country_of_res_'United Kingdom'", "ethnicity_'Middle Eastern '", 'ethnicity_asian', 'country_of_res_Egypt', 'country_of_res_Other', 'country_of_res_Jordan', 'ethnicity_Black', "country_of_res_'United States'", 'ethnicity_Asian'}
MISMATCH in adolescent: {'ethnicity_middle eastern', 'country_of_res_Argentina', 'country_of_res_United Kingdom', 'ethnicity_White European', 'country_of_res_Indonesia', 'ethnicity_south asian', 'ethnicity_South Asian', 'ethnicity_black', 'country_of_res_India', 'ethnicity_White-European', 'country_of_res_Unknown', 'country_of_res_AmericanSamoa', 'ethnicity_Middle Eastern ', 'ethnicity_asian', 'country_of_res_United States', 'country_of_res

In [7]:
all_splits_ok = True
for c in COHORTS:
    n_rows = len(dfs[c])
    for fold in range(N_FOLDS):
        with open(f"{PATH_SPLITS}{c}_fold{fold}.pkl", "rb") as f:
            idx = pickle.load(f)
        train, calib, test = set(idx["train_idx"]), set(idx["calib_idx"]), set(idx["test_idx"])

        overlap = (train & calib) | (train & test) | (calib & test)
        if overlap:
            print(f"OVERLAP in {c} fold{fold}: {len(overlap)} indices shared")
            all_splits_ok = False

        union = train | calib | test
        if len(union) != n_rows:
            print(f"COVERAGE ISSUE in {c} fold{fold}: union covers {len(union)} of {n_rows} rows")
            all_splits_ok = False

print("Split indices check:", "PASSED" if all_splits_ok else "FAILED - see issues above")

Split indices check: PASSED


In [8]:
all_predictions_ok = True
for c in COHORTS:
    for model in ALL_MODELS:
        for fold in range(N_FOLDS):
            for slice_name in ["calib", "test"]:
                probs_path = f"{PATH_PREDICTIONS}{c}_{model}_fold{fold}_{slice_name}slice_probs.npy"
                labels_path = f"{PATH_PREDICTIONS}{c}_{model}_fold{fold}_{slice_name}slice_labels.npy"

                if not (os.path.exists(probs_path) and os.path.exists(labels_path)):
                    print(f"MISSING: {c}/{model}/fold{fold}/{slice_name}")
                    all_predictions_ok = False
                    continue

                probs = np.load(probs_path)
                labels = np.load(labels_path)

                if probs.shape != labels.shape:
                    print(f"SHAPE MISMATCH: {c}/{model}/fold{fold}/{slice_name} - probs{probs.shape} vs labels{labels.shape}")
                    all_predictions_ok = False

                if probs.min() < 0 or probs.max() > 1:
                    print(f"RANGE ISSUE: {c}/{model}/fold{fold}/{slice_name} - min={probs.min()}, max={probs.max()}")
                    all_predictions_ok = False

print("Predictions check:", "PASSED" if all_predictions_ok else "FAILED - see issues above")

Predictions check: PASSED


In [9]:
import re

pattern = re.compile(r"^(" + "|".join(COHORTS) + r")_(" + "|".join(ALL_MODELS) + r")_(" + "|".join(ALL_METHODS) + r")_fold(\d)_testslice_probs\.npy$")

all_files_ok = True
if os.path.exists(PATH_CALIBRATED):
    files = os.listdir(PATH_CALIBRATED)
    for f in files:
        if not pattern.match(f):
            print(f"PATTERN MISMATCH: {f}")
            all_files_ok = False
    print(f"Checked {len(files)} files in calibrated_predictions/")
else:
    print("calibrated_predictions/ folder doesn't exist yet")

print("Filename pattern check:", "PASSED" if all_files_ok else "FAILED - see issues above")

Checked 324 files in calibrated_predictions/
Filename pattern check: PASSED
